# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook applies the same careful reading to two findings from the FlyRank research paper and then to **my own Week-5 model**. The paper was written for a broad audience and states its own standards; the point here is not to grade it but to practise the next level of rigor, and to hold my own work to it first.

**Plan.** (1) Two paper findings and the methodology question I would ask about each, with a small check on my own data for each. (2) My Week-5 model re-run under an honest split, before and after. (3) A leakage audit of my features, a test that proves my harness can detect leakage, and real failure examples. (4) My boldest Week-5 claim, rewritten to what the evidence supports. Claims use the words observed, measured, directional and decision-support.

## 0. Setup

Loads the starter file (30,000 pages, 32 clients). The outcome, as in Weeks 4 and 5, is `declining` = `trend_direction == "down"` (impressions in the last 30 days more than 20% below the 30 days before). It grades models and is never an input.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isfile("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
        os.chdir("..")

import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
warnings.filterwarnings("ignore")

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score

SEED = 0
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__, "| seed", SEED)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["declining"] = (df["trend_direction"] == "down").astype(int)   # the outcome: grades models, never an input
print(f"{len(df):,} pages, {df['client_id'].nunique()} clients | overall declining rate {df['declining'].mean():.1%}")


scikit-learn 1.9.1 | pandas 3.0.5 | numpy 2.5.3 | seed 0
30,000 pages, 32 clients | overall declining rate 54.2%


## 1. Two paper findings + my methodology questions

I picked one headline finding and one machine-learning appendix result, because they test two different things: how a comparison group is built, and what an accuracy number means. My questions are the ones I would want a reviewer to ask of my own notebook.

**Finding A — "The Freshness Multiplier" (finding 4).** The paper reports that content 365+ days old that was refreshed within 30 days shows about 3.2x the health score (10.7 to 34.5) and about 57x the impressions (71 to 4,039), and says refresh timing is one of the strongest measured levers in the portfolio. I like that the paper already marks the 361+ day freshness bucket as too small to trust (283 rising pages against 1 falling). My questions, in the same spirit:

1. **Which two groups are compared, and how big is each?** Of the 365+ day pages, how many were refreshed and how many were not, from how many brands? The paper's own rule is a minimum of 50 pages per bucket.
2. **Could brand mix explain the gap?** If refreshed pages come mostly from a few brands and the untouched pages from others, a refresh multiplier can be a brand multiplier. A within-brand comparison would separate the two.
3. **Does the measured window come after the refresh?** The impressions are a 90-day total, but a "refreshed in the last 30 days" page was only refreshed during the last third of that window. What do the 60 earlier days look like for the same pages?
4. **Is the label a decision?** Pages that get refreshed are usually chosen. If someone refreshed the pages that already looked promising, the multiplier mixes the effect of the refresh with the reason it was picked.

**Finding B — the growth classifier in the ML appendix ("71% holdout accuracy").** The paper describes a logistic regression separating growing from declining pages and says clearly that it is descriptive. My questions:

1. **What is the base rate?** 71% means little until it sits next to "always guess the bigger class". With the paper's own finding-1 counts (74,187 rising against 45,272 falling), always guessing "rising" would score about 62%.
2. **Where does the label come from, and what window do the features use?** Growth comes from a 30-day against previous-30-day impression change. Features such as impressions, clicks and days visible are normally 90-day totals, which contain those same 30 days.
3. **Was the 80/20 split random rows or whole brands?** Pages from one brand share hidden traits, so a random split can reward recognising the brand.

Below, the code backs each question with what the same question looks like in the starter file. I cannot check the paper's own numbers (I only have a 30,000-page slice, not the 341,701-page portfolio), so these are checks of the *shape* of the question, not corrections of the paper.

In [2]:
# Paper finding A (the "freshness multiplier"): redo the paper's own arithmetic, then look at the comparison GROUPS in my file.
print("paper arithmetic: health 10.7 -> 34.5 =", round(34.5 / 10.7, 2), "x   |   impressions 71 -> 4,039 =", round(4039 / 71, 1), "x")

old = df[df["content_age_days"] >= 365].copy()                      # "365+ day content"
old["group"] = np.select(
    [old["days_since_last_update"] <= 30, old["days_since_last_update"] >= 181],
    ["refreshed in the last 30 days", "not updated for 181+ days"], default="updated 31-180 days ago")
grp = old.groupby("group").agg(pages=("content_id", "size"), clients=("client_id", "nunique"),
                               mean_impressions=("impressions_90d", "mean"), median_impressions=("impressions_90d", "median")).round(1)
display(grp)

ref = old[old["group"] == "refreshed in the last 30 days"]
print(f"largest single client holds {ref['client_id'].value_counts(normalize=True).iloc[0]:.0%} of the 'refreshed' pages")
print(f"the 'not updated' group has {int(grp.loc['not updated for 181+ days', 'pages'])} pages from {int(grp.loc['not updated for 181+ days', 'clients'])} client "
      f"- the paper itself sets a minimum of 50 pages per bucket (30 for cross-correlations)")
print(f"ratio of means in my file: {grp.loc['refreshed in the last 30 days', 'mean_impressions'] / grp.loc['not updated for 181+ days', 'mean_impressions']:,.0f}x   "
      f"(ratio of medians: {grp.loc['refreshed in the last 30 days', 'median_impressions'] / grp.loc['not updated for 181+ days', 'median_impressions']:,.0f}x)")

# The only client that has any "not updated" pages: compare inside that one client, so client mix cannot explain the gap.
one = old.loc[old["group"] == "not updated for 181+ days", "client_id"].iloc[0]
print("\nwithin that one client:")
display(old[old["client_id"] == one].groupby("group")["impressions_90d"].agg(pages="size", mean="mean", median="median").round(1))


paper arithmetic: health 10.7 -> 34.5 = 3.22 x   |   impressions 71 -> 4,039 = 56.9 x


                               pages  clients  mean_impressions  \
group                                                             
not updated for 181+ days          5        1               8.2   
refreshed in the last 30 days   5807        9            4968.2   
updated 31-180 days ago          548        4            7499.6   

                               median_impressions  
group                                              
not updated for 181+ days                     2.0  
refreshed in the last 30 days               817.0  
updated 31-180 days ago                    1166.5  

largest single client holds 52% of the 'refreshed' pages
the 'not updated' group has 5 pages from 1 client - the paper itself sets a minimum of 50 pages per bucket (30 for cross-correlations)
ratio of means in my file: 606x   (ratio of medians: 408x)

within that one client:


                               pages    mean  median
group                                               
not updated for 181+ days          5     8.2     2.0
refreshed in the last 30 days    105   226.8    14.0
updated 31-180 days ago          116  1238.8   302.0

**What the freshness question looks like in my file (observed).** The file has 6,360 pages that are 365+ days old. 5,807 were updated in the last 30 days (9 clients; one client holds 52% of them), 548 were updated 31-180 days ago, and only **5 pages from a single client** were not updated for 181+ days. Taken at face value the ratio of mean impressions is about 600x, but it rests on 5 pages from one client, below the paper's own 50-page minimum, so I would not call it a finding. Inside that one client the ratio of means is about 28x, smaller but still resting on 5 pages (second table). This is the reason for questions 1 and 2: the number is only as good as the size and client mix of the comparison group.

Question 3 matters too. Even with a large comparison, a 90-day impression total cannot show an effect of an update made in the last 30 days without also showing the 60 days before it. The cleaner design is the same pages before and after the update, against similar pages that were not updated. Wording I would be comfortable with is "pages updated recently also had higher impressions in this snapshot (observed, directional)", not "refresh multiplies impressions".

In [3]:
# Paper finding B (the growth classifier, "71% holdout accuracy"): what does 71% mean, and does it survive an honest split?
# Same question in my file: up vs down pages only, a logistic regression on features like the paper's list, 80/20 splits repeated 10 times.
ud = df[df["trend_direction"].isin(["up", "down"])].reset_index(drop=True)
y_ud = (ud["trend_direction"] == "up").astype(int).values
print(f"up-vs-down pages in my file: {len(ud):,} from {ud['client_id'].nunique()} clients; share that is 'up' = {y_ud.mean():.1%}  ->  always guessing 'down' scores {1 - y_ud.mean():.1%}")
paper_up, paper_down = 74187, 45272                                  # counts printed in the paper's finding 1
print(f"paper's own finding-1 counts: always guessing 'rising' scores {paper_up / (paper_up + paper_down):.1%}, so 71% would be about {71 - 100 * paper_up / (paper_up + paper_down):.0f} points above that (if the appendix sample looks like it)")

PAPER_LIKE = ["content_age_days", "days_since_last_update", "days_with_impressions", "avg_position", "word_count",
              "impressions_90d", "search_volume", "clicks_90d", "ai_sessions_90d", "sessions_90d"]
NO_TOTALS = ["content_age_days", "days_since_last_update", "avg_position", "word_count", "search_volume"]   # drops the 90-day activity totals
WINDOW_PEEK = PAPER_LIKE + ["impressions_last_30d", "impressions_prev_30d"]                                   # deliberately leaky: the label window itself

def logit():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=2000))

def up_down(cols, grouped, n=10):
    X, out = ud[cols], []
    for s in range(n):
        splitter = GroupShuffleSplit(1, test_size=0.2, random_state=s) if grouped else ShuffleSplit(1, test_size=0.2, random_state=s)
        tr, te = next(splitter.split(X, y_ud, ud["client_id"]))
        if len(set(y_ud[te])) < 2:
            continue
        p = logit().fit(X.iloc[tr], y_ud[tr]).predict_proba(X.iloc[te])[:, 1]
        out.append([accuracy_score(y_ud[te], p > 0.5), max(y_ud[te].mean(), 1 - y_ud[te].mean()), roc_auc_score(y_ud[te], p)])
    return np.mean(out, axis=0)

rows = []
for name, cols in [("paper-like features", PAPER_LIKE), ("without the 90-day activity totals", NO_TOTALS), ("LEAK CHECK: + the two 30-day windows", WINDOW_PEEK)]:
    for grouped in (False, True):
        a, b, c = up_down(cols, grouped)
        rows.append([name, "whole clients held out" if grouped else "random pages", round(a, 3), round(b, 3), round(a - b, 3), round(c, 3)])
display(pd.DataFrame(rows, columns=["features", "split", "accuracy", "always-guess-majority", "accuracy minus majority", "AUC"]))


up-vs-down pages in my file: 20,650 from 30 clients; share that is 'up' = 21.2%  ->  always guessing 'down' scores 78.8%
paper's own finding-1 counts: always guessing 'rising' scores 62.1%, so 71% would be about 9 points above that (if the appendix sample looks like it)


                               features                   split  accuracy  \
0                   paper-like features            random pages     0.794   
1                   paper-like features  whole clients held out     0.803   
2    without the 90-day activity totals            random pages     0.796   
3    without the 90-day activity totals  whole clients held out     0.805   
4  LEAK CHECK: + the two 30-day windows            random pages     0.885   
5  LEAK CHECK: + the two 30-day windows  whole clients held out     0.891   

   always-guess-majority  accuracy minus majority    AUC  
0                  0.790                    0.005  0.641  
1                  0.808                   -0.005  0.563  
2                  0.790                    0.006  0.635  
3                  0.808                   -0.003  0.561  
4                  0.790                    0.095  0.942  
5                  0.808                    0.083  0.928  

**What 71% looks like in my file (measured).** In my file only 21.2% of the up-or-down pages are "up", so always guessing "down" scores about 79%, a very different base rate from the paper's 62%. That is the point of question 1: the same model on a different mix gives a different headline. With paper-like features the logistic regression scores 0.794 accuracy on a random split and 0.803 on held-out clients against 0.790 and 0.808 for always guessing the majority, so accuracy minus majority is between -0.005 and +0.005: no skill by this measure. The AUC (which ignores the base rate) is 0.641 on a random split and 0.563 on held-out clients, so a small ordering signal exists and more than half of it disappears when whole clients are held out (directional; 10 splits).

The leak check is the useful part. Adding the two 30-day windows that define the label lifts accuracy to about 0.89 and AUC to 0.93-0.94, so the harness sees leakage when it is there (a linear model does not read a ratio perfectly, so it does not reach 1.0). That is why question 2 is worth asking about the paper's 90-day features: they are not the 30-day columns themselves, but they contain them. For the paper this is consistent with how it already describes the model (descriptive, exploratory), and a reader could ask for the base rate, the split design and the feature windows as a short addition.

## 2. My model under an honest split (before/after)

**What I re-run.** My Week-5 setup, rebuilt exactly: the same 13,205-page slice (position 20 or better, at least 300 impressions in 90 days, 29 clients), the same Week-4 rule, the same 22 features, logistic regression and random forest (the two strongest readable/strong models from Week 5). The only thing that changes is the split.

- **Before:** a random page-level split (30% of pages held out). Pages from the same client are on both sides, so a model can recognise a client's habits.
- **After:** whole clients held out (30% of clients per split), the Week-5 design, with an assertion that no client is on both sides.

Both use 10 fixed-seed splits and the same metric as Weeks 4-5 (precision@K and AUC), and the rule and random picking are scored on the same test pages. The rule has nothing to learn, so it works as a control: if its numbers barely move between the two splits, the change in the models is about memorising clients, not about easier test sets. (A true time-aware split needs the daily history table, which this starter file does not have, so I use the client-grouped split for this comparison and a timeline-based feature audit in Section 3.)

In [4]:
# My Week-5 setup, rebuilt exactly (same slice, same Week-4 rule, same 22 features) so the before/after is on the same model.
MIN_IMPRESSIONS, MAX_POSITION = 300, 20
elig = df[(df["avg_position"] > 0) & (df["avg_position"] <= MAX_POSITION) & (df["impressions_90d"] >= MIN_IMPRESSIONS)].reset_index(drop=True)
elig["position_band"] = pd.cut(elig["avg_position"], [0, 3, 6, 10, 15, 20], labels=["1-3", "4-6", "7-10", "11-15", "16-20"])
elig["band_median_ctr"] = elig.groupby("position_band", observed=True)["ctr"].transform("median")
elig["ctr_gap"] = (1 - elig["ctr"] / elig["band_median_ctr"]).clip(0, 1)
elig["baseline_score"] = elig["ctr_gap"] * np.log1p(elig["impressions_90d"])          # Week-4 rule, unchanged

for c in ["impressions_90d", "clicks_90d", "sessions_90d", "search_volume", "cpc", "word_count"]:
    elig["log_" + c] = np.log1p(elig[c])
elig["has_keyword_data"] = elig["search_volume"].notna().astype(int)
elig["has_word_count"] = elig["word_count"].notna().astype(int)
elig["is_keyword_article"] = (elig["content_type"] == "keyword article").astype(int)
elig["is_feedly_article"] = (elig["content_type"] == "feedly article").astype(int)
for v in ["informational", "transactional", "commercial"]:
    elig["intent_" + v] = (elig["main_intent"] == v).astype(int)

W5_FEATURES = [
    "log_impressions_90d", "log_clicks_90d", "ctr", "avg_position", "ctr_gap", "days_with_impressions",
    "log_sessions_90d", "days_with_sessions", "engagement_rate", "content_age_days", "days_since_last_update",
    "log_search_volume", "competition", "log_cpc", "log_word_count", "has_keyword_data", "has_word_count",
    "is_keyword_article", "is_feedly_article", "intent_informational", "intent_transactional", "intent_commercial",
]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

KS = (50, 100, 500)
COLS = [f"P@{k}" for k in KS] + ["AUC"]
def metrics(score, y):
    return [precision_at_k(score, y, k) for k in KS] + [roc_auc_score(y, score)]

def make_model(name):
    if name == "logistic regression":
        return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=0.5, max_iter=2000))
    return make_pipeline(SimpleImputer(strategy="median"),
                         RandomForestClassifier(n_estimators=200, min_samples_leaf=20, n_jobs=-1, random_state=SEED))

y_all, groups = elig["declining"].values, elig["client_id"].values
print(f"slice: {len(elig):,} pages from {elig['client_id'].nunique()} clients | base rate {y_all.mean():.3f}")
print(f"baseline P@100 on the whole slice: {precision_at_k(elig['baseline_score'], y_all, 100):.3f}  (Week-4 notebook: 0.720)")


slice: 13,205 pages from 29 clients | base rate 0.607
baseline P@100 on the whole slice: 0.720  (Week-4 notebook: 0.720)


In [5]:
# One evaluation function, used for EVERY comparison in this notebook, so "before" and "after" differ only in the split.
N_SPLITS, TEST_FRACTION = 10, 0.30

def evaluate(cols, grouped, models=("logistic regression", "random forest"), n=N_SPLITS):
    """Per-split metrics for each model, the Week-4 rule and random picking, on the same test pages."""
    rows = []
    for seed in range(n):
        splitter = GroupShuffleSplit(1, test_size=TEST_FRACTION, random_state=seed) if grouped else ShuffleSplit(1, test_size=TEST_FRACTION, random_state=seed)
        tr, te = next(splitter.split(elig, y_all, groups))
        if len(set(y_all[te])) < 2 or len(set(y_all[tr])) < 2:
            continue
        if grouped:
            assert not (set(groups[tr]) & set(groups[te])), "a client is on both sides of the split"
        yt = y_all[te]
        for m in models:
            p = make_model(m).fit(elig.iloc[tr][cols], y_all[tr]).predict_proba(elig.iloc[te][cols])[:, 1]
            rows.append([seed, m] + metrics(p, yt))
        rows.append([seed, "Week-4 rule"] + metrics(elig["baseline_score"].values[te], yt))
        rows.append([seed, "random picking"] + [yt.mean()] * len(KS) + [0.5])
    return pd.DataFrame(rows, columns=["split", "method"] + COLS)

def summary(res):
    mean, sd = res.groupby("method")[COLS].mean(), res.groupby("method")[COLS].std()
    return mean.round(3).astype(str) + " +/- " + sd.round(3).astype(str), mean

def gap_vs_rule(res, model):
    m = res[res["method"] == model].set_index("split")[COLS]
    r = res[res["method"] == "Week-4 rule"].set_index("split")[COLS]
    return m - r

order = ["random picking", "Week-4 rule", "logistic regression", "random forest"]
before = evaluate(W5_FEATURES, grouped=False)      # BEFORE: random page-level split
after = evaluate(W5_FEATURES, grouped=True)        # AFTER: whole clients held out (the Week-5 design)
tb, mb = summary(before)
ta, ma = summary(after)
print("BEFORE - random page-level split (pages from the same client on both sides), mean +/- sd over", before["split"].nunique(), "splits")
display(tb.loc[order])
print("AFTER - whole clients held out, same features, same models, same pages, same metric")
display(ta.loc[order])

cmp_rows = []
for m in ["logistic regression", "random forest"]:
    gb, ga = gap_vs_rule(before, m), gap_vs_rule(after, m)
    cmp_rows.append([m, round(mb.loc[m, "P@50"], 3), round(ma.loc[m, "P@50"], 3), round(mb.loc[m, "AUC"], 3), round(ma.loc[m, "AUC"], 3),
                     f"{gb['P@50'].mean():+.3f}", f"{ga['P@50'].mean():+.3f}", f"{(ga['P@50'] > 0).sum()}/{len(ga)}"])
display(pd.DataFrame(cmp_rows, columns=["model", "P@50 random", "P@50 held-out clients", "AUC random", "AUC held-out clients",
                                         "P@50 gain over rule (random)", "P@50 gain over rule (held-out)", "splits won vs rule (held-out)"]))


BEFORE - random page-level split (pages from the same client on both sides), mean +/- sd over 10 splits


                                P@50            P@100            P@500  \
method                                                                   
random picking       0.604 +/- 0.006  0.604 +/- 0.006  0.604 +/- 0.006   
Week-4 rule           0.726 +/- 0.04  0.722 +/- 0.029  0.738 +/- 0.014   
logistic regression  0.918 +/- 0.027   0.91 +/- 0.026  0.835 +/- 0.018   
random forest          0.9 +/- 0.041  0.919 +/- 0.014  0.862 +/- 0.012   

                                 AUC  
method                                
random picking           0.5 +/- 0.0  
Week-4 rule           0.58 +/- 0.007  
logistic regression  0.667 +/- 0.007  
random forest        0.708 +/- 0.006  

AFTER - whole clients held out, same features, same models, same pages, same metric


                                P@50            P@100            P@500  \
method                                                                   
random picking       0.627 +/- 0.069  0.627 +/- 0.069  0.627 +/- 0.069   
Week-4 rule           0.73 +/- 0.049  0.746 +/- 0.073  0.736 +/- 0.095   
logistic regression  0.808 +/- 0.061   0.81 +/- 0.052  0.739 +/- 0.045   
random forest        0.864 +/- 0.068   0.846 +/- 0.06  0.796 +/- 0.072   

                                 AUC  
method                                
random picking           0.5 +/- 0.0  
Week-4 rule          0.586 +/- 0.024  
logistic regression    0.6 +/- 0.045  
random forest        0.638 +/- 0.022  

                 model  P@50 random  P@50 held-out clients  AUC random  \
0  logistic regression        0.918                  0.808       0.667   
1        random forest        0.900                  0.864       0.708   

   AUC held-out clients P@50 gain over rule (random)  \
0                 0.600                       +0.192   
1                 0.638                       +0.174   

  P@50 gain over rule (held-out) splits won vs rule (held-out)  
0                         +0.078                          8/10  
1                         +0.134                         10/10  

**Reading the before/after (measured, directional).**

- **The rule is the control and it barely moves:** P@50 0.726 vs 0.730 and AUC 0.580 vs 0.586 between the two splits. So the drop in the models is not a harder test set.
- **Logistic regression** falls from P@50 0.918 to 0.808 and AUC 0.667 to 0.600. Its apparent gain over the rule shrinks from +0.192 to +0.078 at P@50 (better than the rule in 8 of 10 splits).
- **Random forest** falls from P@50 0.900 to 0.864 and AUC 0.708 to 0.638. Its gain over the rule goes from +0.174 to +0.134 and it wins 10 of 10 splits.
- So a random split would have overstated the logistic regression's advantage by about 2.5x and the forest's by about 1.3x. Whole-client holdout is the honest number, and it is the design Week 5 reported. This notebook documents the size of the difference. (Week 5 used 20 splits and 300 trees and reported +0.123 for the forest and +0.080 for logistic regression; here 10 splits and 200 trees give +0.134 and +0.078, which is within the split-to-split spread of about 0.05-0.07.)
- My guess is that features like page age and the content mix act as a client fingerprint under a random split. I did not test that directly, so it stays a hypothesis.

The gap between 0.9 and 0.8 is itself a finding about how much memorisation a random split would have hidden.

## 3. Leakage audit

The same hunt as Week 3, now on my final feature set. The skill checklist asks for a drawn timeline: every feature must be knowable before the label window. Here the timeline is short. All numbers are trailing 90-day totals ending at export, and the outcome compares the last 30 days with the 30 days before. So the **whole outcome window sits inside every 90-day total**. Only days 61-90 back are strictly earlier. I can compute that earlier slice as "90-day total minus last 30 days minus previous 30 days" (the two 30-day columns are used only to build that slice, never as features).

```
days back:   90 ............ 61 | 60 ........ 31 | 30 ....... 0
              early window       | previous 30d     | last 30d
              (legal features)   |  \______ label compares these two ______/
              \______________ 90-day totals contain all three ______________/
```

Four steps below: (a) a feature-by-feature timing audit, (b) train with and without the suspects, (c) a deliberate leak to prove the harness can see one, and (d) real failure examples.

In [6]:
# Timeline audit. Every number in this file is a trailing 90-day total ending at export time, and the outcome compares the last 30 days with
# the 30 days before. So the 90-day window CONTAINS the whole outcome window. Only days 61-90 back are strictly before it:
#   early window = 90-day total - last 30 days - previous 30 days.
# (The two 30-day columns are used ONLY to build that earlier slice, never as features themselves.)
for m in ["impressions", "clicks", "sessions"]:
    elig[m + "_early"] = elig[m + "_90d"] - elig[m + "_last_30d"] - elig[m + "_prev_30d"]
    elig["log_" + m + "_early"] = np.log1p(elig[m + "_early"])
elig["ctr_early"] = np.where(elig["impressions_early"] > 0, elig["clicks_early"] / elig["impressions_early"].clip(lower=1) * 100, np.nan)
assert (elig[["impressions_early", "clicks_early", "sessions_early"]] >= 0).all().all(), "an early window is negative: columns do not nest"
print(f"pages with no impressions at all in days 61-90 back: {(elig['impressions_early'] == 0).mean():.1%} of the slice")

timing = pd.DataFrame([
    ["log_impressions_90d, log_clicks_90d, log_sessions_90d", "90-day total", "OVERLAPS the label window", "replace by the days 61-90 slice"],
    ["ctr, ctr_gap (and so the Week-4 rule itself)", "built from 90-day totals", "OVERLAPS the label window", "replace by ctr_early (rule kept as is, as the thing to beat)"],
    ["avg_position", "90-day average", "OVERLAPS (cannot be split)", "drop in the strict set"],
    ["days_with_impressions, days_with_sessions, engagement_rate", "90-day counts / rates", "OVERLAPS the label window", "drop in the strict set"],
    ["days_since_last_update", "state at export time", "CONCURRENT: a refresh can fall inside the outcome window", "test with and without"],
    ["content_age_days", "state at export time", "legal (fixed by the creation date)", "keep"],
    ["search_volume, competition, cpc, word_count, content_type, intent", "page / keyword metadata", "legal (static context)", "keep"],
], columns=["feature(s)", "what it is", "timing verdict", "action"])
display(timing)

STRICT = ["log_impressions_early", "log_clicks_early", "log_sessions_early", "ctr_early", "content_age_days",
          "log_search_volume", "competition", "log_cpc", "log_word_count", "has_keyword_data", "has_word_count",
          "is_keyword_article", "is_feedly_article", "intent_informational", "intent_transactional", "intent_commercial"]
STATIC_ONLY = [c for c in STRICT if "early" not in c]
elig["log_search_volume"] = np.log1p(elig["search_volume"]); elig["log_cpc"] = np.log1p(elig["cpc"])   # same columns as in the Week-5 set
FORBIDDEN = ["trend_pct", "trend_direction", "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d",
             "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d", "declining", "content_id", "client_id"]
assert not set(STRICT) & set(FORBIDDEN), "an outcome or 30-day column is among the strict features"
print(f"strict feature set: {len(STRICT)} features (no 90-day aggregate, no outcome column); static-only set: {len(STATIC_ONLY)} features")


pages with no impressions at all in days 61-90 back: 1.2% of the slice


                                          feature(s)  \
0  log_impressions_90d, log_clicks_90d, log_sessi...   
1       ctr, ctr_gap (and so the Week-4 rule itself)   
2                                       avg_position   
3  days_with_impressions, days_with_sessions, eng...   
4                             days_since_last_update   
5                                   content_age_days   
6  search_volume, competition, cpc, word_count, c...   

                 what it is  \
0              90-day total   
1  built from 90-day totals   
2            90-day average   
3     90-day counts / rates   
4      state at export time   
5      state at export time   
6   page / keyword metadata   

                                      timing verdict  \
0                          OVERLAPS the label window   
1                          OVERLAPS the label window   
2                         OVERLAPS (cannot be split)   
3                          OVERLAPS the label window   
4  CONCURRENT: a refre

strict feature set: 16 features (no 90-day aggregate, no outcome column); static-only set: 12 features


**Timing audit (observed).** Most of my Week-5 features are 90-day aggregates: impressions, clicks, sessions, CTR, the CTR gap that drives my Week-4 rule, average position, days with impressions, days with sessions and engagement. They all contain the outcome window. In Week 5 I wrote this down as a limit ("the click features may be partly mechanical"); here I act on it. `days_since_last_update` is a different case: it does not contain the answer, but an update can happen inside the outcome window, so I test it with and without. Page age, keyword context, word count, content type and intent are fixed or static and are legal.

The strict set has 16 features: activity in days 61-90 only (impressions, clicks, sessions, CTR) plus the static context. 1.2% of the slice had no impressions at all in days 61-90, which the median fill and the missing CTR handle.

In [7]:
# The "train with / train without the suspects" test, all on held-out clients, same 10 splits, same pages, same metric.
sets = {
    "Week-5 features (90-day aggregates included)": W5_FEATURES,
    "strict timeline: days 61-90 activity + static context": STRICT,
    "strict timeline + days_since_last_update": STRICT + ["days_since_last_update"],
    "static context only (no activity history at all)": STATIC_ONLY,
}
audit = {name: evaluate(cols, grouped=True) for name, cols in sets.items()}
rows = []
for name, res in audit.items():
    for m in ["logistic regression", "random forest"]:
        g = gap_vs_rule(res, m)
        mean_m = res[res["method"] == m][COLS].mean()
        rows.append([name, m, round(mean_m["P@50"], 3), round(mean_m["P@100"], 3), round(mean_m["AUC"], 3), f"{g['P@50'].mean():+.3f}", f"{g['AUC'].mean():+.3f}", f"{(g['P@50'] > 0).sum()}/{len(g)}"])
rule_mean = audit[list(sets)[0]].query("method == 'Week-4 rule'")[COLS].mean().round(3)
display(pd.DataFrame(rows, columns=["feature set", "model", "P@50", "P@100", "AUC", "P@50 minus rule", "AUC minus rule", "splits won (P@50)"]))
print("Week-4 rule on the same pages:", rule_mean.to_dict(), "| base rate (random picking): about", round(float(np.mean(y_all)), 3))


                                         feature set                model  \
0       Week-5 features (90-day aggregates included)  logistic regression   
1       Week-5 features (90-day aggregates included)        random forest   
2  strict timeline: days 61-90 activity + static ...  logistic regression   
3  strict timeline: days 61-90 activity + static ...        random forest   
4           strict timeline + days_since_last_update  logistic regression   
5           strict timeline + days_since_last_update        random forest   
6   static context only (no activity history at all)  logistic regression   
7   static context only (no activity history at all)        random forest   

    P@50  P@100    AUC P@50 minus rule AUC minus rule splits won (P@50)  
0  0.808  0.810  0.600          +0.078         +0.014              8/10  
1  0.864  0.846  0.638          +0.134         +0.053             10/10  
2  0.786  0.775  0.609          +0.056         +0.023              7/10  
3  0.806  

Week-4 rule on the same pages: {'P@50': 0.73, 'P@100': 0.746, 'P@500': 0.736, 'AUC': 0.586} | base rate (random picking): about 0.607


**Train with and without the suspects (measured, held-out clients, same 10 splits).**

- With the **Week-5 features** the forest beats the rule by +0.134 at P@50 (10 of 10 splits). With the **strict timeline** features it still beats it, by +0.076 at P@50 (9 of 10) and +0.033 AUC (8 of 10). So roughly 40% of the forest's gain at the top of the list came from, or was helped by, the overlapping 90-day features, and a smaller but consistent gain remains. Logistic regression goes from +0.078 (8 of 10) to +0.056 (7 of 10).
- **Static context alone** (age, keywords, word count, content type, intent) is **worse than the rule** (logistic AUC 0.505, forest 0.538): without activity history there is almost no signal. Whatever the models learn comes from how a page's search visibility has looked, not from what kind of page it is.
- Adding `days_since_last_update` does not help (forest gain +0.034). I keep it out.

One caution on fairness: my Week-4 rule still uses the overlapping 90-day totals, so it benefits from the overlap while the strict model does not. That makes the strict-model-against-rule comparison conservative for the model, but it is not a perfectly like-for-like contest, and I report it as that.

In [8]:
# Is my test harness able to see leakage at all?  Deliberately add the label window (log change between the last two 30-day windows) and watch the score jump.
elig["LEAK_log_change_30d"] = np.log1p(elig["impressions_last_30d"]) - np.log1p(elig["impressions_prev_30d"])   # for this check ONLY
leak = evaluate(STRICT + ["LEAK_log_change_30d"], grouped=True, models=("logistic regression",))
clean = audit["strict timeline: days 61-90 activity + static context"]
cmp = pd.DataFrame({
    "strict features (kept)": clean[clean["method"] == "logistic regression"][COLS].mean(),
    "strict + deliberate leak (thrown away)": leak[leak["method"] == "logistic regression"][COLS].mean()}).round(3).T
display(cmp)
elig.drop(columns=["LEAK_log_change_30d"], inplace=True)
assert "LEAK_log_change_30d" not in elig.columns
print("the leaky column is removed again; every other result in this notebook was produced without it.")

# Population check: the slice itself is chosen with a 90-day number (>= 300 impressions), which includes the outcome window.
print(f"\ndeclining rate: all {len(df):,} pages {df['declining'].mean():.1%} | my slice {y_all.mean():.1%} "
      f"| pages in the slice that had no impressions at all in days 61-90: {(elig['impressions_early'] == 0).mean():.1%}")


                                         P@50  P@100  P@500    AUC
strict features (kept)                  0.786  0.775  0.739  0.609
strict + deliberate leak (thrown away)  1.000  1.000  1.000  1.000

the leaky column is removed again; every other result in this notebook was produced without it.

declining rate: all 30,000 pages 54.2% | my slice 60.7% | pages in the slice that had no impressions at all in days 61-90: 1.2%


**Can my harness see leakage? (measured).** Yes. Adding one column that is the log change between the last two 30-day windows (the outcome's own ingredient) moves the strict logistic regression from P@50 0.786 / AUC 0.609 to 1.000 / 1.000. The column is dropped again right after; none of the other results include it.

**Population check (observed).** The slice keeps pages with at least 300 impressions in 90 days. That is a rule based on a number that includes the outcome window, so the pages I model are partly selected by how they ended up. The declining rate in the slice is 60.7% against 54.2% across all 30,000 pages. I do not know in which direction this biases the model comparison, so I treat it as a disclosed choice: the Week-5 claims apply to "pages that still had at least 300 impressions", not to every page.

In [9]:
# Real failure examples, from the HONEST (strict-timeline, held-out-client) random forest: 5 folds, each fold's clients predicted by a model that never saw them.
oof = np.zeros(len(elig))
for tr, te in GroupKFold(n_splits=5).split(elig, y_all, groups):
    oof[te] = make_model("random forest").fit(elig.iloc[tr][STRICT], y_all[tr]).predict_proba(elig.iloc[te][STRICT])[:, 1]
elig["oof_prob"] = oof
print(f"out-of-fold AUC {roc_auc_score(y_all, oof):.3f} | Week-4 rule AUC on the same pages {roc_auc_score(y_all, elig['baseline_score']):.3f}")

top = elig.sort_values("oof_prob", ascending=False).head(100)
print(f"the model's 100 most-sure pages: {top['declining'].mean():.0%} really declining (base rate {y_all.mean():.0%}); {top['client_id'].nunique()} different clients")
print(f"for comparison the Week-4 rule's top 100: {elig.sort_values('baseline_score', ascending=False).head(100)['declining'].mean():.0%}")

# 1) where it fails: by position band
band = (elig.groupby("position_band", observed=True)
        .apply(lambda g: pd.Series({"pages": len(g), "declining_rate": g["declining"].mean(),
                                    "AUC": roc_auc_score(g["declining"], g["oof_prob"])})).round(3))
display(band)

# 2) where it fails: by client (level vs ordering)
cl = (elig.groupby("client_id")
      .apply(lambda g: pd.Series({"pages": len(g), "declining_rate": g["declining"].mean(), "mean_predicted": g["oof_prob"].mean(),
                                  "within_client_AUC": roc_auc_score(g["declining"], g["oof_prob"]) if (g["declining"].nunique() == 2 and len(g) >= 100) else np.nan})))
print(f"clients whose mean predicted probability is more than 0.15 away from their real declining rate: {(abs(cl['mean_predicted'] - cl['declining_rate']) > 0.15).sum()} of {len(cl)}")
ok = cl["within_client_AUC"].dropna()
print(f"within-client AUC (clients with >= 100 pages): median {ok.median():.3f}, and {(ok < 0.5).sum()} of {len(ok)} clients are ranked WORSE than a coin flip")

# 3) three concrete wrong cases: confident predictions of "declining" that were not
wrong = top[top["declining"] == 0].head(3)
cols = ["oof_prob", "impressions_90d", "impressions_early", "avg_position", "clicks_90d", "days_since_last_update", "content_age_days", "trend_direction", "trend_pct"]
display(wrong[cols].round(2))
print("their clients' declining rates:", ", ".join(f"{cl.loc[c, 'declining_rate']:.2f}" for c in wrong["client_id"]))


out-of-fold AUC 0.599 | Week-4 rule AUC on the same pages 0.581
the model's 100 most-sure pages: 78% really declining (base rate 61%); 8 different clients
for comparison the Week-4 rule's top 100: 72%


                pages  declining_rate    AUC
position_band                               
1-3             507.0           0.750  0.750
4-6            3140.0           0.603  0.630
7-10           4507.0           0.583  0.604
11-15          3037.0           0.608  0.559
16-20          2014.0           0.631  0.566

clients whose mean predicted probability is more than 0.15 away from their real declining rate: 16 of 29
within-client AUC (clients with >= 100 pages): median 0.594, and 0 of 15 clients are ranked WORSE than a coin flip


      oof_prob  impressions_90d  impressions_early  avg_position  clicks_90d  \
3173      0.92              740                235          12.4           0   
2536      0.91             2615               1179           9.3           1   
447       0.91             5993               3191          17.4           4   

      days_since_last_update  content_age_days trend_direction  trend_pct  
3173                      20               141          stable      -10.2  
2536                      13               228              up      126.4  
447                      104               148          stable       -9.8  

their clients' declining rates: 0.60, 0.48, 0.62


**Real failure examples, from the honest model (observed).**

- **Overall:** out-of-fold AUC is 0.599 against 0.581 for the Week-4 rule, almost the same. At the top of the list it does better: of the 100 pages it is most sure about, 78% really declined (base rate 61%), against 72% for the rule's top 100.
- **By position:** AUC is 0.75 for positions 1-3 (507 pages), 0.63 and 0.60 for 4-6 and 7-10 (where most pages sit), and only about 0.56-0.57 for positions 11-20. It is a weaker sorter exactly where there are most pages.
- **By client:** for 16 of 29 clients, the average predicted probability is more than 0.15 away from the client's true declining rate, so the model is not a judge of how a whole client is doing. Within clients that have 100+ pages the median AUC is 0.594 and none of the 15 clients is ranked worse than chance. It is a modest sorter inside a client, not a thermometer across clients.
- **Three confident wrong cases:** the pages with probabilities 0.91-0.92 that were not "down" had zero to four clicks and 740-5,993 impressions. Two were "stable" with a change of about -10% (so, near misses to the -20% cut-off, which a hard threshold counts as plain errors) and one was *up* by +126%. Their clients decline 48-62% of the time, so a client effect does not explain them. A near miss on a hard threshold is not the same kind of error as a wrong direction, and a human reviewer would learn more from the first than from the second.

## 4. Claim rewrite

My boldest Week-5 sentence (the decision-support summary):

> "On held-out clients a random forest ranked pages better than my Week-4 rule (about +0.12 at P@50, 19 of 20 splits) ... I would treat it as a better first-pass sorter, not as a forecast."

Most of it holds, but it goes further than the evidence in four places. The audit above is what lets me say which:

| Phrase in my Week-5 claim | Problem found in this audit | What the evidence supports |
|---|---|---|
| "+0.12 at P@50" for the forest | Came from features that contain the outcome window; with only features that come before it the gain is +0.076 (9 of 10 splits), AUC +0.033 | Quote the strict-timeline number as the headline; keep the Week-5 number as an upper bound |
| "ranked pages better than my rule" | The rule also uses the overlapping totals, so the contest is not like for like; AUC is 0.62 vs 0.59 | "ranked somewhat better at the top of the list", not "better" in general |
| "on held-out clients" | 29 clients, a few very large, one snapshot, a slice chosen with a number from the outcome window | Keep, but add the client count, the snapshot and the slice rule |
| "better first-pass sorter" | True mostly for pages in positions 1-3 and inside a client; weak for positions 11-20 and not a judge of a whole client | Say where it works and where it does not |

In [10]:
# The numbers my rewritten claim is allowed to use - all pulled from the held-out-client, strict-timeline run above.
strict = audit["strict timeline: days 61-90 activity + static context"]
w5 = audit["Week-5 features (90-day aggregates included)"]
rows = []
for label, res in [("Week-5 features, held-out clients", w5), ("strict timeline, held-out clients", strict)]:
    g = gap_vs_rule(res, "random forest")
    rows.append([label, f"{g['P@50'].mean():+.3f}", f"{(g['P@50'] > 0).sum()} of {len(g)}", f"{g['P@100'].mean():+.3f}", f"{g['AUC'].mean():+.3f}", f"{(g['AUC'] > 0).sum()} of {len(g)}",
                 round(res[res['method'] == 'random forest']['AUC'].mean(), 3)])
display(pd.DataFrame(rows, columns=["random forest minus Week-4 rule", "P@50 gain", "splits won (P@50)", "P@100 gain", "AUC gain", "splits won (AUC)", "forest AUC"]))

Path("work/outputs").mkdir(parents=True, exist_ok=True)
_ = Path("work/outputs/w06_audit_results.json").write_text(json.dumps({
    "splits": N_SPLITS, "test_fraction": TEST_FRACTION, "seed": SEED,
    "before_random_split": mb.round(4).to_dict(orient="index"), "after_heldout_clients": ma.round(4).to_dict(orient="index"),
    "strict_timeline_heldout": strict.groupby("method")[COLS].mean().round(4).to_dict(orient="index")}, indent=2))
print("saved work/outputs/w06_audit_results.json")


     random forest minus Week-4 rule P@50 gain splits won (P@50) P@100 gain  \
0  Week-5 features, held-out clients    +0.134          10 of 10     +0.100   
1  strict timeline, held-out clients    +0.076           9 of 10     +0.045   

  AUC gain splits won (AUC)  forest AUC  
0   +0.053          9 of 10       0.638  
1   +0.033          8 of 10       0.619  

saved work/outputs/w06_audit_results.json


**Rewritten claim (decision-support, directional).**

> In this one snapshot (29 clients, 13,205 pages that kept at least 300 impressions), a random forest trained only on features that come before the outcome window and tested on clients it had not seen ranked pages somewhat better than my Week-4 rule at the top of the list: precision@50 +0.076 (better in 9 of 10 splits) and AUC 0.62 against 0.59. The advantage is directional and based on 10 re-splits of a small number of clients; it is largest for pages ranked 1-3 and weak for positions 11-20, the model cannot tell how a whole client is doing, and a random split would have made the same model look much stronger (P@50 0.90 instead of 0.86). I would use it as a decision-support list for a human reviewer to look at first, not as a forecast and not as a reason why pages decline.

**Also changed.** The random-split numbers are shown as the "before", not as results. For the paper, I would add three things to the appendix: the base rate next to the 71%, whether the split held out brands, and which window the features come from.

**What this audit did not do (limits).** (1) No true time split: the starter file is one snapshot, and the daily warehouse table is the proper next step for a forward-window test. (2) The early window is only 30 days wide and a page's early traffic may partly reflect mean reversion, which I did not separate. (3) Ten splits, 29 clients; the standard deviations are wide (0.05-0.07 at P@50) and the 8 or 9 of 10 "wins" are a count, not a significance test. (4) I compared several feature sets on the same splits, so the best-looking row is a little optimistic. (5) Versions of scikit-learn and pandas are printed in Section 0; tree numbers can move a few points between versions.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.